## Generate Virginia pnode names and ids to subset PJM lmp data
                                                                                                                                    
1. Load sample PJM CSV: data/raw/pjm_lmp_data/rt_hrl_lmps.csv                                                                       
    - This contains 24 hours of lmp data for March 15, 2026 - so is expected to contain all currently active pnode ids as of March  
15 2026.                                                                                                                            
    - Find all unique pnode names and ids and subset these.                                                                         
                                                            
2. Load:                                                                                                                            
    - us states boundaries data: data/raw/census_data/tl_2025_us_state/tl_2025_us_state.shp
    - electric substations data: data/raw/electricSubstations/Shapefile/electric_substations_hifld_v4_FieldMods.shp                 
                                                                                                                                    
3. Filter only for substations that are geographically in Virginia. Filter out substations that cannot be used for analysis (ie.    
that have an unknown name and therefore cannot be matched to a pnode). Filter to SUBSTATION type only.                              
                                                                                                                                    
4. Match these substations to the pnodes in the PJM data by name. Use fuzzy matching and then use manual checks to confirm.
                                                                                                                                    
    - Note: PJM pnode id $\neq$ HIFLD_ID                                                                                            
                                        
5. Merge. Then create a file (va_pnode_ids_final_full) that includes (joined by the matched pnode names):                           
    substation_name; latitude; longitude; pnode_match; pnode_id; type                                                               
                                                                                                                                    
    - Summarise the PJM data type unique values (used for data download later)                                                      
                                                                            
6. Then: subset this to pnode_id only, excluding AGGREGATE type nodes, to create: pnode_ids_final  

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import geopandas as gpd

### Load and Inspect Sample LMP Hourly Data (data from one day)

In [2]:
#Set up file paths
sys.path.append(str(Path.cwd().parents[1] / "src"))
from utils.paths import DATA_DIR

In [ ]:
#Check structure of hourly lmp data
sample_hourly_lmp = pd.read_csv(DATA_DIR /  "raw" / "pjm_lmp_data" / "rt_hrl_lmps.csv")
print(sample_hourly_lmp.columns.tolist())


['datetime_beginning_utc', 'datetime_beginning_ept', 'pnode_id', 'pnode_name', 'voltage', 'equipment', 'type', 'zone', 'system_energy_price_rt', 'total_lmp_rt', 'congestion_price_rt', 'marginal_loss_price_rt', 'row_is_current', 'version_nbr']


In [4]:
#Inspect
sample_hourly_lmp.head()

,datetime_beginning_utc,datetime_beginning_ept,pnode_id,pnode_name,voltage,equipment,type,zone,system_energy_price_rt,total_lmp_rt,congestion_price_rt,marginal_loss_price_rt,row_is_current,version_nbr
0,3/15/2026 4:00:00 AM,3/15/2026 12:00:00 AM,1,PJM-RTO,NaN,NaN,ZONE,NaN,21.69,21.710752,0.013905,0.011014,True,1
1,3/15/2026 4:00:00 AM,3/15/2026 12:00:00 AM,3,MID-ATL/APS,NaN,NaN,ZONE,NaN,21.69,22.857885,0.689557,0.482495,True,1
2,3/15/2026 4:00:00 AM,3/15/2026 12:00:00 AM,48592,ALDENE,230 KV,T-10,LOAD,PSEG,21.69,23.920000,1.500000,0.730000,True,1
3,3/15/2026 4:00:00 AM,3/15/2026 12:00:00 AM,48593,ALDENE,230 KV,T-20,LOAD,PSEG,21.69,23.920000,1.500000,0.730000,True,1
4,3/15/2026 4:00:00 AM,3/15/2026 12:00:00 AM,48594,ATHENIA,26 KV,AB GRP,LOAD,PSEG,21.69,24.460000,1.790000,0.980000,True,1


In [5]:
#Identify unique zone and type values
# Unique zone values
unique_zones = sorted(sample_hourly_lmp["zone"].dropna().unique())

# Unique type values
unique_types = sorted(sample_hourly_lmp["type"].dropna().unique())

print("Zones:", unique_zones)
print("Types:", unique_types)

Zones: ['AECO', 'AEP', 'APS', 'ATSI', 'BGE', 'COMED', 'CPL', 'DAY', 'DEOK', 'DOM', 'DPL', 'DUKE', 'DUQ', 'EKPC', 'EXTERNAL', 'JCPL', 'METED', 'OVEC', 'PECO', 'PENELEC', 'PEPCO', 'PPL', 'PSEG', 'RECO']
Types: ['AGGREGATE', 'EHV', 'EXT', 'GEN', 'HUB', 'INTERFACE', 'LOAD', 'RESIDUAL_METERED_EDC', 'ZONE']


In [ ]:
#Check if we can filter only by bus nodes (not aggregate or hub)
busnodes = sample_hourly_lmp[sample_hourly_lmp["type"].isin(["GEN", "LOAD"])].copy()
print("Bus Type Nodes:", busnodes["type"].unique())

Bus Type Nodes: <ArrowStringArray>
['LOAD', 'GEN']
Length: 2, dtype: str


In [8]:
# Unique pnode names
unique_pnode_names = (sample_hourly_lmp["pnode_name"]
    .dropna()
    .unique()
)

print(unique_pnode_names[:10])  # preview
print("Number of unique pnode names:", len(unique_pnode_names))

<ArrowStringArray>
[    'PJM-RTO', 'MID-ATL/APS',      'ALDENE',     'ATHENIA',    'BELLEVIL',
    'BENNETTS',    'BRIDGEWA',     'CLIF PS',     'FOUNDRY',    'GREENBRO']
Length: 10, dtype: str
Number of unique pnode names: 6656


### Load Substation and State Geo Data

In [ ]:
#Load
substation_data = gpd.read_file(DATA_DIR / "raw" / "electricSubstations" / "Shapefile" / "electric_substations_hifld_v4_FieldMods.shp")

census_data = gpd.read_file(DATA_DIR / "raw" / "census_data" / "tl_2025_us_state" / "tl_2025_us_state.shp")

for name, gdf in {"substation": substation_data, "census": census_data}.items():
    print(f"\n--- {name} ---")
    print(gdf.columns.tolist())


--- substation ---
['Name', 'HIFLD_Name', 'HIFLD_ID', 'city', 'City_Prop', 'zip', 'type', 'Type_Prop', 'status', 'Stat_Prop', 'county', 'Count_Prop', 'countyfips', 'StateShort', 'StateFull', 'country', 'latitude', 'longitude', 'naics_code', 'naics_desc', 'source', 'sourcedate', 'val_method', 'val_date', 'lines', 'LineTxt', 'max_volt', 'CurrentMax', 'min_volt', 'CurrentMin', 'max_infer', 'min_infer', 'MinVoltCat', 'geometry']

--- utility ---
['OBJECTID', 'ID', 'NAME', 'ADDRESS', 'CITY', 'STATE', 'ZIP', 'TYPE', 'COUNTRY', 'SOURCE', 'SOURCEDATE', 'VAL_DATE', 'WEBSITE', 'REGULATED', 'CNTRL_AREA', 'PLAN_AREA', 'HOLDING_CO', 'SUMMR_PEAK', 'WINTR_PEAK', 'SUMMER_CAP', 'WINTER_CAP', 'NET_GEN', 'PURCHASED', 'NET_EX', 'RETAIL_MWH', 'WSALE_MWH', 'TOTAL_MWH', 'TRANS_MWH', 'CUSTOMERS', 'YEAR', 'Shape__Are', 'Shape__Len', 'NameProp', 'TypeProp', 'CityProp', 'CntrlProp', 'SimpleType', 'HoldCoProp', 'NameFinal', 'TypeAdj', 'geometry']

--- census ---
['REGION', 'DIVISION', 'STATEFP', 'STATENS', 'GEOI

In [ ]:
#Check CRSs of all layers to ensure they match
print(census_data.crs)
print(substation_data.crs)

EPSG:4269
EPSG:4326
EPSG:3857


In [ ]:
#Update all layers to a common CRS - 4326 is fine for filtering/subsetting
TARGET_CRS = "EPSG:4326"

census_data = census_data.to_crs(TARGET_CRS)
substation_data = substation_data.to_crs(TARGET_CRS)

In [34]:
virginia = census_data[census_data["STUSPS"] == "VA"]                           
                                                                                
#Filter substations to Virginia                                                 
virginia_substations = substation_data[substation_data["StateShort"] == "VA"]   
print(f"Number of substations in Virginia: {len(virginia_substations)}")        

# Inspect substation types and name quality                                   
print("\nSubstation types:")                                                  
print(virginia_substations["type"].value_counts())                            
                                                                                
print("\nSample names (SUBSTATION type):")                                    
print(virginia_substations[virginia_substations["type"] ==                    
"SUBSTATION"]["Name"].sample(30, random_state=42).tolist()) 
                                                                            

Number of substations in Virginia: 1432

Substation types:
type
SUBSTATION    1097
TAP            307
DEAD END        22
RISER            6
Name: count, dtype: int64

Sample names (SUBSTATION type):
['Unknown114445', 'Halifax', 'Unknown114457', 'Fieldale', 'Bristers', 'Unknown114528', 'Celanese', 'Unknown114452', 'Unknown114629', 'Candlers Mountain', 'Roanoke', 'Mountain Run', 'Clifton', 'Garner DP', 'Unknown171567', 'Fork Pickett', 'NIVO', 'Kidds Store', 'Jefferson Street', 'Arlington', 'Unknown167395', 'Hayfield', 'Salem Electric Department', 'Pleasant Valley', 'Unknown114516', 'Unknown116486', 'Unknown172768', 'River Road', 'Unknown117039', 'Danville']


In [39]:
# Filter to SUBSTATION type only
virginia_substations = virginia_substations[virginia_substations["type"] == "SUBSTATION"]
print(f"\nSubstations after type filter: {len(virginia_substations)}")

# Filter out any with unknown in the name
virginia_substations_clean = virginia_substations[
    ~virginia_substations["Name"].str.upper().str.contains("UNKNOWN", na=False)
]
print(f"Substations after removing unknown names: {len(virginia_substations_clean)}")

# Spatial join to confirm within Virginia boundary
virginia_substations_geo = gpd.sjoin(
    virginia_substations_clean,
    virginia[["geometry"]],
    how="inner",
    predicate="within"
)
print(f"Substations within Virginia boundary: {len(virginia_substations_geo)}")  


Substations after type filter: 1097
Substations after removing unknown names: 592
Substations within Virginia boundary: 592


In [40]:
# Preview both name formats
print("Substation names (sample):")
print(virginia_substations_clean["Name"].head(20).tolist())

print("\nPJM pnode names (sample):")
print(sample_hourly_lmp["pnode_name"].unique()[:20])

Substation names (sample):
['Gore', 'Imboden', 'Dorchester', 'Richlands', 'Claypool Hill', 'Sandy Ridge', 'Jewell Ridge', 'Jewell Branch', 'Buckhorn', 'Lonesome Pine', 'Clinchfield', 'Fletchers Ridge', 'Hales Branch', 'Jewell Smokeless C and C Coal Co', 'Shack Mills Station', 'Kean Mtn and Grassy Creek', 'Buchanan Generation LLC', 'Looney Creek', 'Grundy', 'Freemont']

PJM pnode names (sample):
<ArrowStringArray>
[    'PJM-RTO', 'MID-ATL/APS',      'ALDENE',     'ATHENIA',    'BELLEVIL',
    'BENNETTS',    'BRIDGEWA',     'CLIF PS',     'FOUNDRY',    'GREENBRO',
    'HAWTHORN',    'HILLSDAL',     'HOBOKEN',      'KILMER',    'KINGSLAN',
      'LEONIA',    'LAKENELS',     'MINUEST',     'NEWP PS',     'PENHORN']
Length: 20, dtype: str


In [44]:
# Fuzzy matching substation names to pnode names

from rapidfuzz import process, fuzz

pnode_names = sample_hourly_lmp["pnode_name"].unique().tolist()

# One match per substation, include coordinates from the start
results = []
for _, row in virginia_substations_geo.iterrows():
    match, score, idx = process.extractOne(
        row["Name"].upper(),
        [p.upper() for p in pnode_names],
        scorer=fuzz.token_sort_ratio
    )
    results.append({
        "substation_name": row["Name"],
        "latitude": row["latitude"],
        "longitude": row["longitude"],
        "pnode_match": pnode_names[idx],
        "score": score
    })

matches_df = pd.DataFrame(results)

print(f"Total substations: {len(matches_df)}")
print(matches_df.sort_values("score", ascending=False).to_string())

Total substations: 592
                            substation_name   latitude  longitude          pnode_match       score
296                                 Dryburg  36.820047 -78.736010              DRYBURG  100.000000
307                            Joshua Falls  37.406399 -79.077043         JOSHUA FALLS  100.000000
305                                   Monel  37.456485 -79.124261                MONEL  100.000000
304                                 Boxwood  37.593455 -79.032425              BOXWOOD  100.000000
302                                  Midway  38.026582 -78.707160               MIDWAY  100.000000
301                                Grottoes  38.266069 -78.812819             GROTTOES  100.000000
300                                   Dooms  38.107035 -78.849097                DOOMS  100.000000
298                                  Clover  36.866211 -78.706544               CLOVER  100.000000
291                                   Wurno  37.074693 -80.717405                WURNO

In [ ]:
# Export to Excel
output_path = DATA_DIR / "processed" / "preprocessing" / "pnode_data" / "substation_pnode_matches.xlsx"

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    matches_df[matches_df["score"] >= 90].sort_values("score", ascending=False).to_excel(
        writer, sheet_name="high_90plus", index=True
    )
    matches_df[(matches_df["score"] >= 70) & (matches_df["score"] < 90)].sort_values("score", ascending=False).to_excel(
        writer, sheet_name="medium_70to90", index=True
    )
    matches_df[(matches_df["score"] >= 50) & (matches_df["score"] < 70)].sort_values("score", ascending=False).to_excel(
        writer, sheet_name="low_50to70", index=True
    )
    matches_df[matches_df["score"] < 50].sort_values("score", ascending=False).to_excel(
        writer, sheet_name="very_low_sub50", index=True
    )

print(f"Exported to {output_path}")
print(f"  high_90plus:    {len(matches_df[matches_df['score'] >= 90])} rows")
print(f"  medium_70to90:  {len(matches_df[(matches_df['score'] >= 70) & (matches_df['score'] < 90)])} rows")
print(f"  low_50to70:     {len(matches_df[(matches_df['score'] >= 50) & (matches_df['score'] < 70)])} rows")
print(f"  very_low_sub50: {len(matches_df[matches_df['score'] < 50])} rows")

Exported to /Users/elenamurray/Documents/Documents/Repositories/MDS_THESIS/Master-Thesis-2026/data/processed/substation_pnode_matches_2.xlsx
  high_90plus:    239 rows
  medium_70to90:  205 rows
  low_50to70:     130 rows
  very_low_sub50: 18 rows


The file is then manually reviewed, by checking coordinates against the GridStatus.io nodlal price map. Manual checks done by:
- Asking Claude AI to flag any obvious errors and reviewing the flagged rows
- Reviewing all cases with fuzzy match scores less than 60 
- Reviewing duplicate cases with two matches
- Doing manual checks

In [ ]:
# Load the final assessed match file
match_final = pd.read_csv(DATA_DIR / "processed" / "preprocessing" / "pnode_data" / "Substation_Pnode_Match_Final.csv")
print(f"Number of substations to match: ({len(match_final['substation_name'])})")

# Get unique pnode_id / pnode_name pairs for all types
pnode_lookup = sample_hourly_lmp[["pnode_id", "pnode_name",
    "type"]].drop_duplicates()

# Join pnode_id onto the match file
match_with_ids = match_final.merge(
    pnode_lookup,
    left_on="pnode_match",
    right_on="pnode_name",
    how="left"
)

# Check unmatched pnode names
unmatched = match_with_ids[match_with_ids["pnode_id"].isna()]
print(f"Unmatched: ({len(unmatched)})")
print(f"Number of pnode ids: ({len(match_with_ids['pnode_id'])})")

#Inspect
match_with_ids.head(20)

Number of substations to match: (499)
Unmatched: (0)
Number of pnode ids: (1190)


,Unnamed: 0,substation_name,latitude,longitude,pnode_match,pnode_id,pnode_name,type
0,553,Abingdon,36.720046,-82.007967,ABINGDON,32410481,ABINGDON,LOAD
1,553,Abingdon,36.720046,-82.007967,ABINGDON,32410483,ABINGDON,LOAD
2,553,Abingdon,36.720046,-82.007967,ABINGDON,45565837,ABINGDON,LOAD
3,119,Acca,37.579939,-77.477550,ACCA,34885183,ACCA,LOAD
4,119,Acca,37.579939,-77.477550,ACCA,34885185,ACCA,LOAD
5,119,Acca,37.579939,-77.477550,ACCA,34885187,ACCA,LOAD
6,119,Acca,37.579939,-77.477550,ACCA,34885189,ACCA,LOAD
7,119,Acca,37.579939,-77.477550,ACCA,34885191,ACCA,LOAD
8,60,VMEA Peaking Gen,38.739983,-77.509157,AIRPORT,1248985238,AIRPORT,LOAD
9,405,Alpine,37.336270,-77.281097,ALLIED,34885193,ALLIED,LOAD


In [ ]:
# Keep one row per unique pnode_id, retaining substation info                 
pnode_ids_final = (
    match_with_ids.dropna(subset=["pnode_id"])                                
    .drop_duplicates(subset=["pnode_id"])
    [["substation_name", "latitude", "longitude", "pnode_match", "pnode_id",  
"type"]]        
    .reset_index(drop=True)
)

print(f"Rows: {len(pnode_ids_final)}")
print("\nType breakdown:")
print(pnode_ids_final["type"].value_counts())

# Save full list (including all types)
pnode_ids_final.to_csv(
    DATA_DIR / "processed" / "preprocessing" / "pnode_data" / "va_pnode_ids_final_full.csv",
index=False
)
print(f"\nSaved {len(pnode_ids_final)} pnode_ids (full)")

# Filter out AGGREGATE type for the download list
pnode_ids_no_agg = pnode_ids_final[pnode_ids_final["type"] != "AGGREGATE"].reset_index(drop=True)
print(f"Rows after removing AGGREGATE: {len(pnode_ids_no_agg)}")

# Save filtered pnode_ids for LMP data download
pnode_ids_no_agg[["pnode_id"]].to_csv(
    DATA_DIR / "processed" / "preprocessing" / "pnode_data" / "va_pnode_ids_final.csv",
index=False
)
print(f"Saved {len(pnode_ids_no_agg)} pnode_ids (no AGGREGATE)")

Rows: 1152

Type breakdown:
type
LOAD         899
GEN          204
EHV           36
AGGREGATE     13
Name: count, dtype: int64

Saved 1152 pnode_ids (full)
Rows after removing AGGREGATE: 1139
Saved 1139 pnode_ids (no AGGREGATE)


Note: where multiple substations share a pnode_id, this keeps the first one.

Each pnode name maps to multiple pnode_ids because a single substation can
have multiple measurement points, each with its own unique id. 